# 10. 전압 이상치 탐지 (U1, U2, U3)

## 이상치 기준
- 물리적 기준: 유럽 전압 규격 EN 50160 기준 정격 230V ± 10% = 207~253V 범위 초과
- 통계적 기준: 계량기별 일별 평균값 기준 평균 ± 3σ 초과

In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

# 유럽 전압 규격 EN 50160: 230V ± 10%
V_LOWER = 207.0
V_UPPER = 253.0

ALL_METERS = [
    'H1.Z10', 'H1.Z11', 'H1.Z12', 'H1.Z13', 'H1.Z14', 'H1.Z15', 'H1.Z16',
    'H1.Z17', 'H1.Z18', 'H1.Z19', 'H1.Z20', 'H1.Z21', 'H1.Z22', 'H1.Z23',
    'H1.Z24', 'H1.Z25', 'H1.Z26', 'H1.Z27', 'H1.Z28', 'H1.Z29', 'H1.Z310',
    'H1.ZE20', 'H2.T.Z30', 'H2.T.Z31', 'H2.T.Z32', 'H2.T.Z33', 'H2.T.Z34',
    'H2.Z311', 'H2.Z35', 'H2.Z64', 'H2.Z65', 'H2.Z66', 'H2.Z67', 'H2.Z68',
    'H2.Z69', 'H2.Z70', 'H2.ZE64', 'H2.ZE65', 'H2.ZE66', 'H2.ZE67', 'H2.ZE74',
    'H3.Z312', 'H3.Z40', 'H3.Z41', 'H3.Z42', 'H4.Z50', 'H4.Z51', 'H4.ZE50',
    'H4.ZE51', 'V.Z81', 'V.Z82', 'V.Z84', 'V.ZE84'
]

save_dir = ROOT / 'outputs/tables/anomaly'
save_dir.mkdir(parents=True, exist_ok=True)
print('설정 완료')

설정 완료


In [2]:
def fetch_daily(meter_urn, measurement):
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    df['day'] = pd.to_datetime(df['day'])
    return df


def detect_anomaly(meter, measurement):
    df = fetch_daily(meter, measurement)
    if df.empty:
        return None

    results = []

    # 1. 물리적 기준: EN 50160 207~253V 범위 초과
    physical = df[(df['min_val'] < V_LOWER) | (df['max_val'] > V_UPPER)].copy()
    physical['anomaly_type'] = '물리적이상(전압범위초과)'
    physical['criterion'] = f'min_val < {V_LOWER}V 또는 max_val > {V_UPPER}V (EN 50160 230V±10%)'
    if len(physical) > 0:
        results.append(physical)

    # 2. 통계적 기준: 평균 ± 3σ
    mean_val = df['avg_val'].mean()
    std_val  = df['avg_val'].std()
    upper = mean_val + 3 * std_val
    lower = mean_val - 3 * std_val
    stat = df[(df['avg_val'] > upper) | (df['avg_val'] < lower)].copy()
    stat['anomaly_type'] = '통계적이상(3sigma)'
    stat['criterion'] = f'mean={mean_val:.2f}, sigma={std_val:.2f}, lower={lower:.2f}, upper={upper:.2f}'
    if len(stat) > 0:
        results.append(stat)

    if not results:
        return None

    result = pd.concat(results).drop_duplicates('day').sort_values('day')
    result['meter'] = meter
    result['measurement'] = measurement
    return result[['meter', 'measurement', 'day', 'min_val', 'max_val', 'avg_val', 'anomaly_type', 'criterion']]

In [3]:
all_results = []

for meter in ALL_METERS:
    for meas in ['U1', 'U2', 'U3']:
        result = detect_anomaly(meter, meas)
        if result is not None and len(result) > 0:
            print(f'{meter} {meas}: {len(result)}건')
            all_results.append(result)

if all_results:
    final = pd.concat(all_results, ignore_index=True)
    final.to_csv(save_dir / 'anomaly_voltage.csv', index=False)
    print(f'\n총 {len(final)}건 저장 완료')
else:
    print('이상치 없음')

/tmp/ipykernel_109661/1368278103.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 U1: 12건


/tmp/ipykernel_109661/1368278103.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 U2: 9건


/tmp/ipykernel_109661/1368278103.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


...

/tmp/ipykernel_109661/1368278103.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z84 U3: 8건
V.ZE84 U1: 4건


/tmp/ipykernel_109661/1368278103.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_109661/1368278103.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.ZE84 U2: 4건
V.ZE84 U3: 4건

총 1180건 저장 완료


/tmp/ipykernel_109661/1368278103.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


In [4]:
if all_results:
    summary = final.groupby(['meter', 'measurement', 'anomaly_type']).agg(
        건수=('day', 'count'),
        시작일=('day', 'min'),
        종료일=('day', 'max'),
        min_val=('min_val', 'min'),
        max_val=('max_val', 'max'),
        criterion=('criterion', 'first')
    ).reset_index()
    summary.to_csv(save_dir / 'anomaly_voltage_summary.csv', index=False)
    print(summary.to_string())

        meter measurement   anomaly_type  건수        시작일        종료일     min_val     max_val                                                 criterion
0      H1.Z10          U1  통계적이상(3sigma)  12 2018-01-16 2023-12-30  223.950000  235.946962       mean=229.89, sigma=0.93, lower=227.10, upper=232.67
1      H1.Z10          U2  통계적이상(3sigma)   9 2018-01-16 2023-12-30  223.876667  236.705299       mean=229.71, sigma=0.93, lower=226.91, upper=232.51
2      H1.Z10          U3  통계적이상(3sigma)  13 2018-01-16 2023-12-30  224.292500  236.584197       mean=230.31, sigma=1.00, lower=227.30, upper=233.31
3      H1.Z11          U1  통계적이상(3sigma)  12 2018-01-16 2023-12-30  223.969167  235.968624       mean=229.91, sigma=0.92, lower=227.13, upper=232.68
4      H1.Z11          U2  통계적이상(3sigma)   9 2018-01-16 2023-12-30  223.817500  236.552090       mean=229.64, sigma=0.93, lower=226.86, upper=232.42
5      H1.Z11          U3  통계적이상(3sigma)  13 2018-01-16 2023-12-30  224.366667  236.616191       mean=230.